# Data Wrangling

In this notebook, I continued on the analysis of the data collected on the game: Arc Raiders through YouTube comments.

In [28]:
import numpy as np
import pandas as pd
import re
from datetime import datetime, timedelta
from scipy import stats
import os

# For text preprocessing (for RoBERTa later)
import warnings
warnings.filterwarnings('ignore')

In [29]:
# Load the dataset
data = pd.read_csv('data/comments_data.csv')
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 210486 entries, 0 to 210485
Data columns (total 27 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   comment_id         210486 non-null  str    
 1   text               210400 non-null  str    
 2   comment_date       210486 non-null  str    
 3   author_hash        210486 non-null  str    
 4   parent_id          46219 non-null   str    
 5   last_updated_at    210486 non-null  str    
 6   video_id           210486 non-null  str    
 7   likes              210486 non-null  int64  
 8   video_title        210486 non-null  str    
 9   video_date         210486 non-null  str    
 10  channel_id         210486 non-null  str    
 11  keyword_matched    210486 non-null  str    
 12  video_description  192158 non-null  str    
 13  char_count         210486 non-null  int64  
 14  word_count         210486 non-null  int64  
 15  avg_word_length    210486 non-null  float64
 16  has_url      

## 1. Data Cleaning

### 1.1. Handling Missing Values

In [30]:
data.isna().sum()

comment_id                0
text                     86
comment_date              0
author_hash               0
parent_id            164267
last_updated_at           0
video_id                  0
likes                     0
video_title               0
video_date                0
channel_id                0
keyword_matched           0
video_description     18328
char_count                0
word_count                0
avg_word_length           0
has_url                   0
has_mention               0
has_hashtag               0
exclamation_count         0
question_count            0
emoji_count               0
newline_count             0
uppercase_ratio           0
language                  0
day_of_week               0
day_num                   0
dtype: int64

In [31]:
data[data['video_description'].notna()]

,comment_id,text,comment_date,author_hash,parent_id,last_updated_at,video_id,likes,video_title,video_date,...,has_mention,has_hashtag,exclamation_count,question_count,emoji_count,newline_count,uppercase_ratio,language,day_of_week,day_num
0,Ugzs5vXYaTEQlg7hTRN4AaABAg,The Wizard101 music makes this EXTRA magical,2026-03-04 07:00:34,52a22e83b1bd3e805c364011f296008a1a93d8529226eb...,NaN,2026-03-04 07:00:34.000000,dHR35IlXVMw,0,they just accepted their fate... Arc Raiders t...,2026-03-04 06:53:00,...,False,False,0,0,0,0,0.159091,en,Wednesday,2
1,UgznazUWJimUWAlgECJ4AaABAg,ゲームとしては楽しいんですけどね、、、。\n久々プレイしたら倒された後、わざと蘇生してきて立...,2026-03-04 05:44:15,82da67e6bd090a6fdba53fac768cb634a49227fa53e080...,NaN,2026-03-04 05:44:15.000000,s5J5qOzrh4c,0,【ARC Raiders】ワイプされて、今期はPvPマッチに入りたいさとE,2026-03-04 03:24:07,...,False,False,0,0,0,3,0.000000,ja,Wednesday,2
2,UgzxVDhlr7zFzhIW5NB4AaABAg,今のマッチングシステムになんの不満もない。というかこれでいいじゃん。戦いたくない時はファーム...,2026-03-04 05:41:45,3ddcf20a92386fde845a86b47ba2eb6abd63deb9fafe3a...,NaN,2026-03-04 05:42:09.000000,s5J5qOzrh4c,0,【ARC Raiders】ワイプされて、今期はPvPマッチに入りたいさとE,2026-03-04 03:24:07,...,False,False,0,0,0,0,0.017391,ja,Wednesday,2
3,UgzxVDhlr7zFzhIW5NB4AaABAg.ATvNppIC6uPATvPZgnwO9H,価値観の違いでしかないから納得して欲しいわけではないけど、常に友好的か非友好的なプレイヤー混...,2026-03-04 05:56:53,b30a97042f17443c49bf17c1d5209539211e4456886d2b...,UgzxVDhlr7zFzhIW5NB4AaABAg,2026-03-04 05:56:53.000000,s5J5qOzrh4c,1,【ARC Raiders】ワイプされて、今期はPvPマッチに入りたいさとE,2026-03-04 03:24:07,...,False,False,0,0,0,1,0.023669,ja,Wednesday,2
4,UgzxVDhlr7zFzhIW5NB4AaABAg.ATvNppIC6uPATvVuXezW4Y,​@6quickman いえいえ！ご丁寧にありがとうございます。わかります。中間な感じのマッ...,2026-03-04 06:52:18,3ddcf20a92386fde845a86b47ba2eb6abd63deb9fafe3a...,UgzxVDhlr7zFzhIW5NB4AaABAg,2026-03-04 06:53:05.000000,s5J5qOzrh4c,0,【ARC Raiders】ワイプされて、今期はPvPマッチに入りたいさとE,2026-03-04 03:24:07,...,True,False,0,0,0,0,0.000000,ja,Wednesday,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
210481,UgxsjQf5cGAgBgzmECp4AaABAg.ATQbgbporRLATTQwCU_dbl,Or none,2026-02-20 23:50:52,de7ee45e5f1438e81fbc09feda3a3a7d9f2e3811bd0676...,UgxsjQf5cGAgBgzmECp4AaABAg,2026-02-20 23:50:52.000000,FYz0dMtT4b4,1,ARC Raiders Is Considering Adjusting Certain S...,2026-02-19 21:30:09,...,False,False,0,0,0,0,0.142857,it,Friday,4
210482,UgxsjQf5cGAgBgzmECp4AaABAg.ATQbgbporRLATTtcBHv4Ip,This is why I took security lockers. Actually ...,2026-02-21 04:10:16,c734b5dcbbc810c9340a76e133b5f35e587f5283214265...,UgxsjQf5cGAgBgzmECp4AaABAg,2026-02-21 04:10:16.000000,FYz0dMtT4b4,26,ARC Raiders Is Considering Adjusting Certain S...,2026-02-19 21:30:09,...,False,False,0,0,0,0,0.029412,en,Saturday,5
210483,UgxsjQf5cGAgBgzmECp4AaABAg.ATQbgbporRLATUpx7QwDVp,"Real ass shit twin, useless skill tree",2026-02-21 12:57:25,06c12d28ed9223307e880cf84bd9f0826e418acd3ba5f1...,UgxsjQf5cGAgBgzmECp4AaABAg,2026-02-21 12:57:25.000000,FYz0dMtT4b4,0,ARC Raiders Is Considering Adjusting Certain S...,2026-02-19 21:30:09,...,False,False,0,0,0,0,0.026316,en,Saturday,5
210484,UgxsjQf5cGAgBgzmECp4AaABAg.ATQbgbporRLATVHMJedYtd,"Like slip and slide, it seems to do nothing ac...",2026-02-21 17:05:40,6887e91804ca6ec750623ed563dfc009de332162fa777a...,UgxsjQf5cGAgBgzmECp4AaABAg,2026-02-21 17:05:40.000000,FYz0dMtT4b4,6,ARC Raiders Is Considering Adjusting Certain S...,2026-02-19 21:30:09,...,False,False,0,0,0,0,0.015625,en,Saturday,5


In [32]:
# Filter out rows with missing video descriptions and comments
filtered_data = data[data['video_description'].notna() & data['text'].notna()]
filtered_data.info()

<class 'pandas.DataFrame'>
Index: 192077 entries, 0 to 210485
Data columns (total 27 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   comment_id         192077 non-null  str    
 1   text               192077 non-null  str    
 2   comment_date       192077 non-null  str    
 3   author_hash        192077 non-null  str    
 4   parent_id          42578 non-null   str    
 5   last_updated_at    192077 non-null  str    
 6   video_id           192077 non-null  str    
 7   likes              192077 non-null  int64  
 8   video_title        192077 non-null  str    
 9   video_date         192077 non-null  str    
 10  channel_id         192077 non-null  str    
 11  keyword_matched    192077 non-null  str    
 12  video_description  192077 non-null  str    
 13  char_count         192077 non-null  int64  
 14  word_count         192077 non-null  int64  
 15  avg_word_length    192077 non-null  float64
 16  has_url           

In [33]:
# Even if not null, some video descriptions, text or video_title might be empty strings. Let's check for that.
#filtered_data['video_description'].apply(lambda x: len(x.strip()) == 0).sum()
#filtered_data['text'].apply(lambda x: len(x.strip()) == 0).sum()
#filtered_data['video_title'].apply(lambda x: len(x.strip()) == 0).sum()
filtered_data = filtered_data[~(filtered_data['video_description'].apply(lambda x: len(x.strip()) == 0) | 
                                filtered_data['text'].apply(lambda x: len(x.strip()) == 0) | 
                                filtered_data['video_title'].apply(lambda x: len(x.strip()) == 0))]
filtered_data.shape

(192077, 27)

In [34]:
data = filtered_data.copy()

### 1.2. Standardizing Datetime

In [35]:
data['comment_date'] = pd.to_datetime(data['comment_date'])
data['video_date'] = pd.to_datetime(data['video_date'])
data['last_updated_at'] = pd.to_datetime(data['last_updated_at'])

In [36]:
# Sort by comment date for time series analysis
data = data.sort_values('comment_date').reset_index(drop=True)

print("Date columns converted:")
print(f"  Comment date range: {data['comment_date'].min()} to {data['comment_date'].max()}")
print(f"  Video date range: {data['video_date'].min()} to {data['video_date'].max()}")
print(f"  Total time span: {(data['comment_date'].max() - data['comment_date'].min()).days} days")

Date columns converted:
  Comment date range: 2025-11-11 18:49:44 to 2026-03-04 07:47:21
  Video date range: 2025-11-11 18:49:04 to 2026-03-04 07:01:32
  Total time span: 112 days


## 2. Defining Features

### 2.1. Event Mention Detection

Based on Arc Raiders timeline and major announcements:
- **GenAI Voice Announcement**: When Embark Studios announced AI-generated voices
- **Business Model Change**: When they announced changing from free-to-play to paid
- **Game Launch**: Official release date

These dates will be used to calculate temporal features and decay functions.

In [37]:
MAJOR_EVENTS = {
    'game_announcement': pd.Timestamp('2021-12-10'),
    'genai_announcement': pd.Timestamp('2024-05-15'),
    'business_model_change': pd.Timestamp('2024-08-20'),
    'game_launch': pd.Timestamp('2025-10-30'),
    'early_access': pd.Timestamp('2025-11-15'),
}
print("Major Event Timeline:")
print("=" * 60)
for event, date in MAJOR_EVENTS.items():
    print(f"  {event:25s}: {date.strftime('%Y-%m-%d')}")
print("=" * 60)

Major Event Timeline:
  game_announcement        : 2021-12-10
  genai_announcement       : 2024-05-15
  business_model_change    : 2024-08-20
  game_launch              : 2025-10-30
  early_access             : 2025-11-15


Create binary features to detect if comments mention specific events.
This helps us identify which comments are directly responding to announcements.

In [38]:
EVENT_KEYWORDS = {
    'game_announcement': [
        'announcement', 'announced', 'reveal', 'revealed', 'teaser', 'teased',
        'first look', 'sneak peek', 'game announcement', 'game reveal'
    ],
    'genai': [
        'ai voice', 'ai-generated', 'artificial intelligence', 'generated voice',
        'ai audio', 'synthetic voice', 'ai narration', 'ai dialogue',
        'voice ai', 'ai-voiced', 'ai acting', 'machine voice'
    ],
    'business_model': [
        'free to play', 'f2p', 'freemium', 'paid game', 'pay to play', 'p2p',
        'price', 'cost', 'buying', 'purchase', 'buy the game', 'not free',
        'charging', 'monetization', 'business model', 'payment model'
    ],
    'launch': [
        'launch', 'release', 'released', 'launching', 'came out', 'available',
        'live', 'went live', 'early access'
    ]
}

In [39]:
def detect_event_mention(text, keywords):
    """Detect if text mentions any of the keywords"""
    if pd.isna(text):
        return False
    text_lower = text.lower()
    return any(keyword in text_lower for keyword in keywords)

# Create event mention columns
data['mentions_genai'] = data['text'].apply(
    lambda x: detect_event_mention(x, EVENT_KEYWORDS['genai'])
)
data['mentions_business_model'] = data['text'].apply(
    lambda x: detect_event_mention(x, EVENT_KEYWORDS['business_model'])
)
data['mentions_launch'] = data['text'].apply(
    lambda x: detect_event_mention(x, EVENT_KEYWORDS['launch'])
)

In [40]:
# Summary of event mentions
print("Event Mention Detection Results:")
print("=" * 60)
print(f"Comments mentioning GenAI:           {data['mentions_genai'].sum():>8,} ({data['mentions_genai'].mean()*100:>5.2f}%)")
print(f"Comments mentioning Business Model:  {data['mentions_business_model'].sum():>8,} ({data['mentions_business_model'].mean()*100:>5.2f}%)")
print(f"Comments mentioning Launch:          {data['mentions_launch'].sum():>8,} ({data['mentions_launch'].mean()*100:>5.2f}%)")
print(f"Comments mentioning any event:       {(data['mentions_genai'] | data['mentions_business_model'] | data['mentions_launch']).sum():>8,}")
print("=" * 60)

Event Mention Detection Results:
Comments mentioning GenAI:                 92 ( 0.05%)
Comments mentioning Business Model:     1,536 ( 0.80%)
Comments mentioning Launch:             2,921 ( 1.52%)
Comments mentioning any event:          4,473


### 2.2. Temporal Distance from Events

We calculate how many days each comment was posted relative to each major event.
This is crucial for measuring sentiment decay over time.

In [41]:
# Calculate days from each event
for event_name, event_date in MAJOR_EVENTS.items():
    col_name = f'days_from_{event_name}'
    data[col_name] = (data['comment_date'] - event_date).dt.days

In [42]:
# Display temporal features
print("Temporal Distance Features Created:")
print("=" * 40)
for event_name in MAJOR_EVENTS.keys():
    col_name = f'days_from_{event_name}'
    if col_name in data.columns:
        print(f"\n{event_name}:")
        print(f"  Min: {data[col_name].min():>6.0f} days (before event)")
        print(f"  Max: {data[col_name].max():>6.0f} days (after event)")
        print(f"  Comments before event: {(data[col_name] < 0).sum():>8,}")
        print(f"  Comments after event:  {(data[col_name] >= 0).sum():>8,}")

Temporal Distance Features Created:

game_announcement:
  Min:   1432 days (before event)
  Max:   1545 days (after event)
  Comments before event:        0
  Comments after event:   192,077

genai_announcement:
  Min:    545 days (before event)
  Max:    658 days (after event)
  Comments before event:        0
  Comments after event:   192,077

business_model_change:
  Min:    448 days (before event)
  Max:    561 days (after event)
  Comments before event:        0
  Comments after event:   192,077

game_launch:
  Min:     12 days (before event)
  Max:    125 days (after event)
  Comments before event:        0
  Comments after event:   192,077

early_access:
  Min:     -4 days (before event)
  Max:    109 days (after event)
  Comments before event:    5,946
  Comments after event:   186,131


### 2.3 Engagement Related Features

Here we create engagement-related features using like_count and other metrics.
These will be used for popularity-weighted sentiment scores.

In [43]:
data['has_likes'] = data['likes'] > 0
data['like_count_log'] = np.log1p(data['likes'])  # Log transform for skewed distribution

In [44]:
data['engagement_tier'] = pd.cut(
    data['likes'],
    bins=[-1, 0, 5, 20, 100, float('inf')],
    labels=['no_likes', 'low_engagement', 'medium_engagement', 'high_engagement', 'viral'])

In [45]:
data['comment_latency_days'] = (data['comment_date'] - data['video_date']).dt.total_seconds() / (24 * 3600)
data['comment_latency_hours'] = (data['comment_date'] - data['video_date']).dt.total_seconds() / 3600

In [46]:
data['latency_category'] = pd.cut(
    data['comment_latency_days'],
    bins=[-float('inf'), 0, 1, 7, 30, float('inf')],
    labels=['before_upload', 'same_day', 'within_week', 'within_month', 'after_month'])

In [47]:
print("Engagement Features Summary:")
print("=" * 40)
print(f"Comments with likes:        {data['has_likes'].sum():>8,} ({data['has_likes'].mean()*100:>5.2f}%)")
print(f"Mean like count:            {data['likes'].mean():>12.2f}")
print(f"Median like count:          {data['likes'].median():>12.2f}")
print(f"Max like count:             {data['likes'].max():>12.0f}")
print(f"\nEngagement Tier Distribution:")
for tier in data['engagement_tier'].cat.categories:
    count = (data['engagement_tier'] == tier).sum()
    print(f"  {tier:20s}: {count:>8,} ({count/len(data)*100:>5.2f}%)")

Engagement Features Summary:
Comments with likes:          49,673 (25.86%)
Mean like count:                   11.23
Median like count:                  0.00
Max like count:                    31801

Engagement Tier Distribution:
  no_likes            :  142,404 (74.14%)
  low_engagement      :   38,576 (20.08%)
  medium_engagement   :    6,111 ( 3.18%)
  high_engagement     :    3,086 ( 1.61%)
  viral               :    1,900 ( 0.99%)


## 3. Text Preprocessing

Preparing the text to make it suitable for sentiment analysis with RoBERTa model.

In [48]:
def preprocess_for_roberta(text):
    """
    Preprocess text for RoBERTa sentiment analysis.
    - Replace URLs with [URL] token
    - Replace @mentions with [USER] token
    - Remove excessive whitespace
    """
    if pd.isna(text):
        return ""
    
    # Replace URLs
    text = re.sub(r'http\S+|www\S+', '[URL]', text)
    
    # Replace @mentions
    text = re.sub(r'@\w+', '[USER]', text)
    
    # Clean whitespace
    text = ' '.join(text.split())
    
    return text

In [49]:
data['text_preprocessed'] = data['text'].apply(preprocess_for_roberta)

In [50]:
# Check preprocessing
print("Text Preprocessing Complete:")
print("=" * 60)
print(f"Original text with URLs:     {data['has_url'].sum():>6,}")
print(f"Original text with mentions: {data['has_mention'].sum():>6,}")
print(f"\nSample preprocessed texts:")
print("-" * 60)
for i, row in data[data['has_url'] | data['has_mention']].head(3).iterrows():
    print(f"\nOriginal:     {row['text'][:100]}...")
    print(f"Preprocessed: {row['text_preprocessed'][:100]}...")

Text Preprocessing Complete:
Original text with URLs:        121
Original text with mentions: 11,846

Sample preprocessed texts:
------------------------------------------------------------

Original:     @Yuuhpif I see another raider PK’ing its on sight 😂 but if you’re just grinding bench/scrappy upgrad...
Preprocessed: [USER] I see another raider PK’ing its on sight 😂 but if you’re just grinding bench/scrappy upgrades...

Original:     @RoxxSermI hope they understood & said something like “you were my brother”😭...
Preprocessed: [USER] hope they understood & said something like “you were my brother”😭...

Original:     ​@rockydhaliwal15maybe not some strange things happen here...
Preprocessed: ​[USER] not some strange things happen here...


## 4. Adding context

Adding context from the video_description, as well as the top-level comment for replies, gives a **Conversational NLP** edge to my project. Even though it is not relevant for my RoBERTa-based model, it works great for the LLMs I am going to work with.

In [54]:
data['video_context'] = data['video_description'].str[:500]

In [55]:
# Creating the Parent Text Mapping first
text_map = data.set_index('comment_id')['text'].to_dict()

# Mapping parent texts to replies
data['parent_text'] = data['parent_id'].map(text_map)

## 5. Filtering the data for analysis

### 5.1 Keeping only the high quality data

In [59]:
# High-quality English comments only
data_clean = data[
    (data['language'] == 'en') &
    (data['word_count'] >= 3) &
    (~data['text'].isna())
].copy()

In [60]:
# Bot Removal (Deduplicate identical text)
data_clean = data_clean.drop_duplicates(subset=['text'], keep='first')
data_clean.info()

<class 'pandas.DataFrame'>
Index: 143065 entries, 1 to 192075
Data columns (total 44 columns):
 #   Column                           Non-Null Count   Dtype         
---  ------                           --------------   -----         
 0   comment_id                       143065 non-null  str           
 1   text                             143065 non-null  str           
 2   comment_date                     143065 non-null  datetime64[us]
 3   author_hash                      143065 non-null  str           
 4   parent_id                        29931 non-null   str           
 5   last_updated_at                  143065 non-null  datetime64[us]
 6   video_id                         143065 non-null  str           
 7   likes                            143065 non-null  int64         
 8   video_title                      143065 non-null  str           
 9   video_date                       143065 non-null  datetime64[us]
 10  channel_id                       143065 non-null  str       

### 5.2 Trimming the unnecessary columns from the dataset

In [ ]:
column_list = data_clean.columns.tolist()
print(column_list)

['comment_id', 'text', 'comment_date', 'author_hash', 'parent_id', 'last_updated_at', 'video_id', 'likes', 'video_title', 'video_date', 'channel_id', 'keyword_matched', 'video_description', 'char_count', 'word_count', 'avg_word_length', 'has_url', 'has_mention', 'has_hashtag', 'exclamation_count', 'question_count', 'emoji_count', 'newline_count', 'uppercase_ratio', 'language', 'day_of_week', 'day_num', 'mentions_genai', 'mentions_business_model', 'mentions_launch', 'days_from_game_announcement', 'days_from_genai_announcement', 'days_from_business_model_change', 'days_from_game_launch', 'days_from_early_access', 'has_likes', 'like_count_log', 'engagement_tier', 'comment_latency_days', 'comment_latency_hours', 'latency_category', 'text_preprocessed', 'video_context', 'parent_text']


In [65]:
columns_to_keep = [
    'comment_id', 'parent_id', 'parent_text',  # Conversation context
    'text_preprocessed',                       # Model input
    'video_context', 'video_id',               # Video context
    'comment_date', 'author_hash',             # Temporal/Identity
    'days_from_genai_announcement', 
    'days_from_business_model_change', 
    'days_from_game_launch',                   # Analysis variables
    'mentions_genai', 'mentions_business_model', 
    'likes', 'like_count_log']                 # Engagement

In [66]:
data_final = data_clean[columns_to_keep].copy()

In [67]:
DATA_DIR = "data"
os.makedirs(DATA_DIR, exist_ok=True)

data_final.to_csv(os.path.join(DATA_DIR, 'comments_for_analysis.csv'), index=False)
print(f"Dataset prepared with {len(data_final)} rows.")

Dataset prepared with 143065 rows.
